# Qualitative Analysis: Before/After Comparison & Error Taxonomy

This notebook produces:
1. **Before/After Image Grids** — side-by-side comparison of baseline Janus-Pro vs. RL-trained model
2. **Error Taxonomy** — classify and count error types (Missing Object / Wrong Count / Wrong Attribute / Wrong Relation)
3. **Per-Category Score Tables** — quantitative scores via CLIP + VLM

---

## Step 0: Configuration

**Edit the variables below before running.** All model/API settings are here.

In [ ]:
import os

# ============================================================
# >>> EDIT THESE BEFORE RUNNING <<<
# ============================================================

# SiliconFlow API Key (get yours at https://cloud.siliconflow.cn/)
SILICONFLOW_API_KEY = "sk-xxx"  # <-- Replace with your key
os.environ["SILICONFLOW_API_KEY"] = SILICONFLOW_API_KEY

# VLM model for evaluation
# Options: "Qwen/Qwen2.5-VL-32B-Instruct" (paid, best)
#          "THUDM/GLM-4.1V-9B-Thinking" (free, lower quality)
VLM_MODEL = "Qwen/Qwen2.5-VL-32B-Instruct"

# Baseline model (do not change unless using a different base)
BASELINE_MODEL = "deepseek-ai/Janus-Pro-1B"

# Trained LoRA weights path
# After cloning the repo, the default path is:
LORA_WEIGHTS_PATH = "assets/trained_lora"
# If using checkpoint from training outputs, change to e.g.:
# LORA_WEIGHTS_PATH = "outputs/grpo_siliconflow_quick/final_checkpoint"
# Or from Google Drive:
# LORA_WEIGHTS_PATH = "/content/drive/MyDrive/T2I-RL/outputs/grpo_full/final_checkpoint"

# Output directory for generated figures (absolute path so it shows in Colab file browser)
OUTPUT_DIR = "/content/qualitative_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Generation settings
NUM_IMAGES_PER_PROMPT = 1  # images per prompt (increase for diversity)
SEED = 42  # for reproducibility

# Use 4-bit quantization? (recommended for Colab free tier / T4)
USE_4BIT = True

print("Configuration loaded.")
print(f"  VLM Model: {VLM_MODEL}")
print(f"  LoRA Path: {LORA_WEIGHTS_PATH}")
print(f"  Output Dir: {OUTPUT_DIR}")

## Step 1: Install Dependencies & Clone Repo

In [ ]:
# Check GPU
!nvidia-smi

# Install dependencies (use Colab's pre-installed PyTorch to avoid version conflicts)
!pip install -q transformers==4.49.0 accelerate safetensors
!pip install -q --no-deps git+https://github.com/deepseek-ai/Janus.git
!pip install -q attrdict sentencepiece open-clip-torch peft bitsandbytes
!pip install -q tqdm Pillow matplotlib seaborn pandas openai

print("\n✓ Dependencies installed")

In [ ]:
# Clone the project repo (if not already cloned)
import os
if not os.path.exists('T2I-RL-Project'):
    !git clone https://github.com/Inoriany/T2I-RL-Project.git
os.chdir('T2I-RL-Project')

import sys
sys.path.insert(0, 'src')

# Ensure output dir exists (absolute path, no need to re-create after chdir)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Working directory: {os.getcwd()}")
print(f"LoRA weights exist: {os.path.exists(LORA_WEIGHTS_PATH)}")

## Step 2: Load Models

Load the baseline Janus-Pro-1B and apply LoRA weights for the trained version.

In [ ]:
import torch
from models.generators import JanusProGenerator, GenerationConfig
from models.reward_models import CLIPRewardModel

# --- Load Baseline Model ---
print("Loading baseline Janus-Pro-1B...")
baseline_gen = JanusProGenerator(
    model_name_or_path=BASELINE_MODEL,
    device="cuda",
    load_in_4bit=USE_4BIT,
)
baseline_gen.load_model()
baseline_gen.eval()
print("✓ Baseline model loaded\n")

# --- Load CLIP Reward Model ---
print("Loading CLIP reward model...")
clip_reward = CLIPRewardModel(device="cuda")
clip_reward.load_model()
print("✓ CLIP reward model loaded")

In [ ]:
# --- Load Trained Model (Baseline + LoRA) ---
print("Loading trained model (Baseline + LoRA)...")
trained_gen = JanusProGenerator(
    model_name_or_path=BASELINE_MODEL,
    device="cuda",
    load_in_4bit=USE_4BIT,
)
trained_gen.load_model()
trained_gen.enable_lora(lora_path=LORA_WEIGHTS_PATH)
trained_gen.eval()
print("✓ Trained model loaded with LoRA weights")

In [ ]:
# --- Setup VLM Client ---
from openai import OpenAI

vlm_client = OpenAI(
    api_key=os.environ["SILICONFLOW_API_KEY"],
    base_url="https://api.siliconflow.cn/v1"
)
print("✓ VLM client ready")

## Step 3: Define Evaluation Prompts

We organize prompts into 5 categories aligned with the Error Taxonomy:
- **Object Presence** (tests Missing Object errors)
- **Counting** (tests Wrong Count errors)
- **Color Binding** (tests Wrong Attribute – color)
- **Attribute Binding** (tests Wrong Attribute – size/shape/texture)
- **Spatial Relations** (tests Wrong Relation errors)

In [ ]:
EVAL_PROMPTS = {
    "object_presence": [
        "a cat and a dog",
        "an apple and a banana and an orange",
        "a car and a bicycle",
        "a book and a pen on a desk",
        "a chair and a table",
        "a cup and a plate on a table",
    ],
    "counting": [
        "two cats",
        "three apples on a plate",
        "four birds sitting on a wire",
        "five flowers in a garden",
        "one dog and two cats",
        "three candles on a birthday cake",
    ],
    "color_binding": [
        "a red apple and a green pear",
        "a blue car parked next to a yellow taxi",
        "a white cat sitting beside a black dog",
        "a purple flower in a pink vase",
        "a red ball and a blue cube",
        "a green frog on a red mushroom",
    ],
    "attribute_binding": [
        "a large elephant and a small mouse",
        "a tall building and a short fence",
        "a fluffy white cat and a sleek black cat",
        "an old wooden chair and a new metal chair",
        "a smooth stone and a rough piece of bark",
        "a thick red book and a thin blue book",
    ],
    "spatial_relations": [
        "a cat sitting on top of a box",
        "a ball under the table",
        "a bird flying above the tree",
        "a dog standing behind a fence",
        "a lamp next to a bed",
        "a painting hanging above the fireplace",
    ],
}

total_prompts = sum(len(v) for v in EVAL_PROMPTS.values())
print(f"Total evaluation prompts: {total_prompts}")
for cat, prompts in EVAL_PROMPTS.items():
    print(f"  {cat}: {len(prompts)} prompts")

## Step 4: Generate Before/After Images

Generate images using both models with the same seed for fair comparison.

In [ ]:
from tqdm.auto import tqdm
from PIL import Image
import time

gen_config = GenerationConfig(
    seed=SEED,
    guidance_scale=5.0,
    num_images_per_prompt=NUM_IMAGES_PER_PROMPT,
)

# Store all results
all_results = {}  # category -> list of {prompt, baseline_img, trained_img, ...}

for category, prompts in EVAL_PROMPTS.items():
    print(f"\n=== Generating: {category} ===")
    all_results[category] = []
    
    for prompt in tqdm(prompts, desc=category):
        # Generate baseline image
        baseline_imgs = baseline_gen.generate(prompt=prompt, config=gen_config)
        
        # Generate trained image (same seed)
        trained_imgs = trained_gen.generate(prompt=prompt, config=gen_config)
        
        # Compute CLIP scores
        baseline_clip = clip_reward.compute_reward(baseline_imgs, [prompt]).rewards.item()
        trained_clip = clip_reward.compute_reward(trained_imgs, [prompt]).rewards.item()
        
        all_results[category].append({
            "prompt": prompt,
            "baseline_img": baseline_imgs[0],
            "trained_img": trained_imgs[0],
            "baseline_clip": baseline_clip,
            "trained_clip": trained_clip,
        })
        
        # Clear GPU cache between prompts
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"\n✓ Generated {total_prompts} pairs of images")

## Step 5: Before/After Comparison Grids

Display side-by-side grids for each category with CLIP score annotations.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import textwrap

def plot_before_after_grid(results_dict, category, save_path=None):
    """
    Plot a before/after comparison grid for a given category.
    Each row: prompt | baseline image (CLIP) | trained image (CLIP)
    """
    items = results_dict[category]
    n = len(items)
    
    fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
    if n == 1:
        axes = axes.reshape(1, -1)
    
    fig.suptitle(f"Before / After Comparison — {category.replace('_', ' ').title()}",
                 fontsize=16, fontweight='bold', y=1.02)
    
    for i, item in enumerate(items):
        prompt_text = textwrap.fill(item['prompt'], width=40)
        
        # Baseline (Before)
        axes[i, 0].imshow(item['baseline_img'])
        axes[i, 0].set_aspect('equal')
        axes[i, 0].set_title(f"Before (CLIP: {item['baseline_clip']:.3f})",
                            fontsize=10, color='red')
        axes[i, 0].set_ylabel(prompt_text, fontsize=9, rotation=0,
                             labelpad=120, ha='right', va='center')
        axes[i, 0].set_xticks([])
        axes[i, 0].set_yticks([])
        
        # Trained (After)
        axes[i, 1].imshow(item['trained_img'])
        axes[i, 1].set_aspect('equal')
        clip_diff = item['trained_clip'] - item['baseline_clip']
        color = 'green' if clip_diff > 0 else 'red'
        axes[i, 1].set_title(
            f"After (CLIP: {item['trained_clip']:.3f}, Δ={clip_diff:+.3f})",
            fontsize=10, color=color)
        axes[i, 1].set_xticks([])
        axes[i, 1].set_yticks([])
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  Saved: {save_path}")
    plt.show()

# Generate grids for each category
for category in EVAL_PROMPTS:

    save_path = os.path.join(OUTPUT_DIR, f"before_after_{category}.png")    
    plot_before_after_grid(all_results, category, save_path=save_path)

In [ ]:
def plot_combined_overview_grid(results_dict, save_path=None):
    """
    Plot a compact overview: 1 example per category, before vs after.
    Good for a paper figure.
    """
    categories = list(results_dict.keys())
    n = len(categories)
    
    fig, axes = plt.subplots(n, 2, figsize=(8, 3.5 * n))
    fig.suptitle("Before vs After Training — Overview",
                 fontsize=16, fontweight='bold', y=1.01)
    
    # Column headers
    axes[0, 0].set_title("Baseline (Before)", fontsize=12, fontweight='bold')
    axes[0, 1].set_title("Trained (After)", fontsize=12, fontweight='bold')
    
    for i, cat in enumerate(categories):
        item = results_dict[cat][0]  # first example from each category
        prompt_text = textwrap.fill(item['prompt'], width=30)
        label = f"[{cat.replace('_', ' ').title()}]\n{prompt_text}"
        
        axes[i, 0].imshow(item['baseline_img'])
        axes[i, 0].set_aspect('equal')
        axes[i, 0].set_ylabel(label, fontsize=8, rotation=0,
                             labelpad=100, ha='right', va='center')
        axes[i, 0].set_xticks([])
        axes[i, 0].set_yticks([])
        clip_b = item['baseline_clip']
        axes[i, 0].text(0.02, 0.02, f"CLIP: {clip_b:.3f}",
                       transform=axes[i, 0].transAxes, fontsize=8,
                       color='white', backgroundcolor='black', va='bottom')
        
        axes[i, 1].set_aspect('equal')
        axes[i, 1].set_xticks([])
        axes[i, 1].set_yticks([])
        clip_t = item['trained_clip']
        diff = clip_t - clip_b
        color = 'lime' if diff > 0 else 'red'
        axes[i, 1].text(0.02, 0.02,
                       f"CLIP: {clip_t:.3f} ({diff:+.3f})",
                       transform=axes[i, 1].transAxes, fontsize=8,
                       color=color, backgroundcolor='black', va='bottom')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f"Saved overview grid: {save_path}")
    plt.show()

plot_combined_overview_grid(all_results,

    save_path=os.path.join(OUTPUT_DIR, "overview_before_after.png"))    save_path=os.path.join(OUTPUT_DIR, "overview_before_after.png"))

## Step 6: CLIP Score Summary Table

Aggregate CLIP scores by category for the report.

In [ ]:
import pandas as pd

# Build summary table
summary_rows = []
for category, items in all_results.items():
    baseline_scores = [item['baseline_clip'] for item in items]
    trained_scores = [item['trained_clip'] for item in items]
    
    avg_b = sum(baseline_scores) / len(baseline_scores)
    avg_t = sum(trained_scores) / len(trained_scores)
    delta = avg_t - avg_b
    
    # Count improvements
    improved = sum(1 for b, t in zip(baseline_scores, trained_scores) if t > b)
    
    summary_rows.append({
        "Category": category.replace('_', ' ').title(),
        "Baseline CLIP": f"{avg_b:.4f}",
        "Trained CLIP": f"{avg_t:.4f}",
        "Delta": f"{delta:+.4f}",
        "Improved": f"{improved}/{len(items)}",
    })

# Overall
all_b = [item['baseline_clip'] for items in all_results.values() for item in items]
all_t = [item['trained_clip'] for items in all_results.values() for item in items]
overall_b = sum(all_b) / len(all_b)
overall_t = sum(all_t) / len(all_t)
overall_improved = sum(1 for b, t in zip(all_b, all_t) if t > b)

summary_rows.append({
    "Category": "**Overall**",
    "Baseline CLIP": f"{overall_b:.4f}",
    "Trained CLIP": f"{overall_t:.4f}",
    "Delta": f"{(overall_t - overall_b):+.4f}",
    "Improved": f"{overall_improved}/{len(all_b)}",
})

df_summary = pd.DataFrame(summary_rows)
print("\n" + "=" * 70)
print("CLIP SCORE SUMMARY")
print("=" * 70)
print(df_summary.to_string(index=False))
df_summary.to_csv(os.path.join(OUTPUT_DIR, "clip_score_summary.csv"), index=False)
print(f"\nSaved to {OUTPUT_DIR}/clip_score_summary.csv")

In [ ]:
# Visualize CLIP score comparison as grouped bar chart
import numpy as np

categories_clean = [cat.replace('_', ' ').title() for cat in EVAL_PROMPTS.keys()]
baseline_avgs = []
trained_avgs = []
for cat in EVAL_PROMPTS.keys():
    b_scores = [item['baseline_clip'] for item in all_results[cat]]
    t_scores = [item['trained_clip'] for item in all_results[cat]]
    baseline_avgs.append(np.mean(b_scores))
    trained_avgs.append(np.mean(t_scores))

x = np.arange(len(categories_clean))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, baseline_avgs, width, label='Baseline', color='#FF6B6B', alpha=0.85)
bars2 = ax.bar(x + width/2, trained_avgs, width, label='Trained (RL)', color='#4ECDC4', alpha=0.85)

ax.set_ylabel('CLIP Score', fontsize=12)
ax.set_title('CLIP Score Comparison: Baseline vs. RL-Trained', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories_clean, rotation=20, ha='right', fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, max(max(baseline_avgs), max(trained_avgs)) * 1.15)

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'clip_score_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Step 7: Error Taxonomy — VLM-based Error Classification

Use the VLM to classify each generated image's errors into the taxonomy:
- **MISSING_OBJECT**: An object in the prompt is not present
- **WRONG_COUNT**: The number of objects is incorrect
- **WRONG_ATTRIBUTE**: Color, size, shape, or texture is wrong
- **WRONG_RELATION**: Spatial relationship is incorrect
- **NO_ERROR**: Image perfectly matches the description

In [ ]:
import json
import re
import base64
from io import BytesIO

def classify_errors_vlm(image, prompt, vlm_client, vlm_model):
    """
    Use VLM to classify errors in a generated image against its prompt.
    Returns a dict with error_types, severity, and details.
    """
    buffer = BytesIO()
    image.save(buffer, format='PNG')
    img_base64 = base64.b64encode(buffer.getvalue()).decode()
    
    eval_prompt = f"""Carefully analyze this generated image against the text description: "{prompt}"

Classify ALL errors you find into these categories:
1. MISSING_OBJECT: An object mentioned in the description is not visible in the image
2. WRONG_COUNT: The number of objects does not match (e.g., asked for 3 but shows 2)
3. WRONG_ATTRIBUTE: A visual attribute is incorrect (wrong color, wrong size, wrong texture, wrong shape)
4. WRONG_RELATION: A spatial relationship is incorrect (wrong position: on/under/above/beside etc.)
5. NO_ERROR: The image matches the description well

Return ONLY a JSON object in this exact format:
{{"error_types": ["ERROR_TYPE1", "ERROR_TYPE2"], "details": "brief explanation of each error", "severity": "high/medium/low/none", "score": 0-10}}

If there are no errors, return: {{"error_types": ["NO_ERROR"], "details": "image matches description", "severity": "none", "score": 10}}"""
    
    try:
        response = vlm_client.chat.completions.create(
            model=vlm_model,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": eval_prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_base64}"}},
                ],
            }],
            max_tokens=400,
        )
        result_text = response.choices[0].message.content
        
        # Parse JSON from response
        json_match = re.search(r'\{[^}]+\}', result_text, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group())
            return parsed
        return {"error_types": ["PARSE_ERROR"], "raw": result_text}
    except Exception as e:
        return {"error_types": ["API_ERROR"], "error": str(e)}

print("✓ Error classifier defined")

In [ ]:
# Run error classification on ALL generated images (both baseline and trained)
error_results = {"baseline": [], "trained": []}

for category, items in all_results.items():
    print(f"\nClassifying errors for: {category}")
    for item in tqdm(items, desc=f"{category}"):
        prompt = item['prompt']
        
        # Classify baseline errors
        baseline_errors = classify_errors_vlm(
            item['baseline_img'], prompt, vlm_client, VLM_MODEL)
        error_results["baseline"].append({
            "prompt": prompt,
            "category": category,
            **baseline_errors,
        })
        time.sleep(0.3)  # Rate limiting
        
        # Classify trained errors
        trained_errors = classify_errors_vlm(
            item['trained_img'], prompt, vlm_client, VLM_MODEL)
        error_results["trained"].append({
            "prompt": prompt,
            "category": category,
            **trained_errors,
        })
        time.sleep(0.3)

print(f"\n✓ Error classification complete")

In [ ]:
from collections import Counter

def count_errors(error_list):
    """Count error types from a list of error results."""
    all_errors = []
    for r in error_list:
        etypes = r.get("error_types", [])
        # Normalize error type names
        for e in etypes:
            e_upper = e.upper().strip()
            if "MISSING" in e_upper:
                all_errors.append("Missing Object")
            elif "COUNT" in e_upper:
                all_errors.append("Wrong Count")
            elif "COLOR" in e_upper or "SIZE" in e_upper or "ATTRIBUTE" in e_upper or "TEXTURE" in e_upper or "SHAPE" in e_upper:
                all_errors.append("Wrong Attribute")
            elif "RELATION" in e_upper or "SPATIAL" in e_upper or "POSITION" in e_upper:
                all_errors.append("Wrong Relation")
            elif "NO_ERROR" in e_upper or "NONE" in e_upper:
                all_errors.append("No Error")
            elif "PARSE_ERROR" in e_upper or "API_ERROR" in e_upper:
                pass  # skip technical errors
            else:
                all_errors.append("Other")
    return Counter(all_errors)

baseline_error_counts = count_errors(error_results["baseline"])
trained_error_counts = count_errors(error_results["trained"])

# Build comparison table
error_categories = ["Missing Object", "Wrong Count", "Wrong Attribute", "Wrong Relation", "No Error"]
n_total = len(error_results["baseline"])

taxonomy_rows = []
for err_type in error_categories:
    b_count = baseline_error_counts.get(err_type, 0)
    t_count = trained_error_counts.get(err_type, 0)
    b_pct = b_count / n_total * 100 if n_total > 0 else 0
    t_pct = t_count / n_total * 100 if n_total > 0 else 0
    delta = t_pct - b_pct
    taxonomy_rows.append({
        "Error Type": err_type,
        "Baseline Count": b_count,
        "Baseline %": f"{b_pct:.1f}%",
        "Trained Count": t_count,
        "Trained %": f"{t_pct:.1f}%",
        "Change": f"{delta:+.1f}%",
    })

df_taxonomy = pd.DataFrame(taxonomy_rows)
print("\n" + "=" * 80)
print("ERROR TAXONOMY TABLE")
print("=" * 80)
print(df_taxonomy.to_string(index=False))
df_taxonomy.to_csv(os.path.join(OUTPUT_DIR, "error_taxonomy.csv"), index=False)
print(f"\nSaved to {OUTPUT_DIR}/error_taxonomy.csv")

In [ ]:
# Visualize Error Taxonomy: Side-by-side bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: grouped bar chart ---
error_types_plot = ["Missing Object", "Wrong Count", "Wrong Attribute", "Wrong Relation"]
b_counts = [baseline_error_counts.get(e, 0) for e in error_types_plot]
t_counts = [trained_error_counts.get(e, 0) for e in error_types_plot]

x = np.arange(len(error_types_plot))
width = 0.35

axes[0].bar(x - width/2, b_counts, width, label='Baseline', color='#FF6B6B', alpha=0.85)
axes[0].bar(x + width/2, t_counts, width, label='Trained', color='#4ECDC4', alpha=0.85)
axes[0].set_xlabel('Error Type')
axes[0].set_ylabel('Count')
axes[0].set_title('Error Counts: Baseline vs Trained', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(error_types_plot, rotation=15, ha='right')
axes[0].legend()

# --- Right: pie charts ---
# Baseline pie
all_error_types = ["Missing Object", "Wrong Count", "Wrong Attribute", "Wrong Relation", "No Error"]
colors_pie = ['#FF6B6B', '#FFA07A', '#FFD93D', '#6BCB77', '#4ECDC4']

b_sizes = [baseline_error_counts.get(e, 0) for e in all_error_types]
t_sizes = [trained_error_counts.get(e, 0) for e in all_error_types]

# Only show non-zero slices
b_labels = [f"{e}\n({c})" for e, c in zip(all_error_types, b_sizes) if c > 0]
b_filtered = [c for c in b_sizes if c > 0]
b_colors = [colors_pie[i] for i, c in enumerate(b_sizes) if c > 0]

t_labels = [f"{e}\n({c})" for e, c in zip(all_error_types, t_sizes) if c > 0]
t_filtered = [c for c in t_sizes if c > 0]
t_colors = [colors_pie[i] for i, c in enumerate(t_sizes) if c > 0]

# Use pie chart in second axes, split into two mini-pies
axes[1].set_visible(False)
ax_b = fig.add_axes([0.55, 0.15, 0.2, 0.7])
ax_t = fig.add_axes([0.78, 0.15, 0.2, 0.7])

if b_filtered:
    ax_b.pie(b_filtered, labels=b_labels, colors=b_colors, autopct='%1.0f%%',
             textprops={'fontsize': 7}, startangle=90)
ax_b.set_title('Baseline', fontsize=10, fontweight='bold')

if t_filtered:
    ax_t.pie(t_filtered, labels=t_labels, colors=t_colors, autopct='%1.0f%%',
             textprops={'fontsize': 7}, startangle=90)
ax_t.set_title('Trained', fontsize=10, fontweight='bold')

plt.savefig(os.path.join(OUTPUT_DIR, 'error_taxonomy_chart.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Per-Category Error Breakdown

Show which error types occur in which prompt categories — before and after training.

In [ ]:
import seaborn as sns

def build_category_error_matrix(error_list, categories):
    """Build a matrix: rows = prompt categories, cols = error types."""
    error_types = ["Missing Object", "Wrong Count", "Wrong Attribute", "Wrong Relation", "No Error"]
    matrix = {}
    for cat in categories:
        cat_results = [r for r in error_list if r.get('category') == cat]
        cat_errors = count_errors(cat_results)
        n_cat = max(len(cat_results), 1)
        matrix[cat.replace('_', ' ').title()] = {
            e: cat_errors.get(e, 0) / n_cat * 100 for e in error_types
        }
    return pd.DataFrame(matrix).T

categories_list = list(EVAL_PROMPTS.keys())

# Baseline heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

df_baseline_matrix = build_category_error_matrix(error_results["baseline"], categories_list)
sns.heatmap(df_baseline_matrix, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[0],
            vmin=0, vmax=100, cbar_kws={'label': '%'})
axes[0].set_title('Error Distribution — Baseline', fontweight='bold')
axes[0].set_ylabel('Prompt Category')

df_trained_matrix = build_category_error_matrix(error_results["trained"], categories_list)
sns.heatmap(df_trained_matrix, annot=True, fmt='.0f', cmap='YlGn_r', ax=axes[1],
            vmin=0, vmax=100, cbar_kws={'label': '%'})
axes[1].set_title('Error Distribution — Trained', fontweight='bold')
axes[1].set_ylabel('Prompt Category')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'error_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step 9: Detailed Per-Prompt Error Report

A table showing each prompt, its before/after error types, and whether training fixed or introduced errors.

In [ ]:
per_prompt_rows = []
for b_res, t_res in zip(error_results["baseline"], error_results["trained"]):
    b_errors = b_res.get("error_types", [])
    t_errors = t_res.get("error_types", [])
    
    # Determine if training helped
    b_has_error = any(e not in ["NO_ERROR", "PARSE_ERROR", "API_ERROR"] for e in b_errors)
    t_has_error = any(e not in ["NO_ERROR", "PARSE_ERROR", "API_ERROR"] for e in t_errors)
    
    if not b_has_error and not t_has_error:
        status = "✓ Both correct"
    elif b_has_error and not t_has_error:
        status = "✓ FIXED by training"
    elif not b_has_error and t_has_error:
        status = "✗ BROKEN by training"
    else:
        # Both have errors
        b_score = b_res.get("score", 5)
        t_score = t_res.get("score", 5)
        if isinstance(b_score, str): b_score = float(b_score)
        if isinstance(t_score, str): t_score = float(t_score)
        if t_score > b_score:
            status = "↑ Improved"
        elif t_score < b_score:
            status = "↓ Degraded"
        else:
            status = "— Same"
    
    per_prompt_rows.append({
        "Category": b_res.get('category', '').replace('_', ' ').title(),
        "Prompt": b_res['prompt'][:50],
        "Baseline Errors": ', '.join(b_errors),
        "Trained Errors": ', '.join(t_errors),
        "Status": status,
    })

df_per_prompt = pd.DataFrame(per_prompt_rows)
print("\n" + "=" * 100)
print("PER-PROMPT ERROR REPORT")
print("=" * 100)
# Pretty-print with max column width
with pd.option_context('display.max_colwidth', 50, 'display.width', 200):
    print(df_per_prompt.to_string(index=False))

# Count status summary
status_counts = df_per_prompt['Status'].value_counts()
print("\n--- Status Summary ---")
for status, count in status_counts.items():
    print(f"  {status}: {count}")

df_per_prompt.to_csv(os.path.join(OUTPUT_DIR, 'per_prompt_error_report.csv'), index=False)
print(f"\nSaved to {OUTPUT_DIR}/per_prompt_error_report.csv")

## Step 10: Save All Results

Bundle everything for the final report.

In [ ]:
# Save comprehensive results to JSON
results_json = {
    "clip_summary": summary_rows,
    "error_taxonomy": taxonomy_rows,
    "baseline_errors": [
        {
            "prompt": r["prompt"],
            "category": r.get("category", ""),
            "error_types": r.get("error_types", []),
            "severity": r.get("severity", ""),
            "details": r.get("details", ""),
            "score": r.get("score", None),
        } for r in error_results["baseline"]
    ],
    "trained_errors": [
        {
            "prompt": r["prompt"],
            "category": r.get("category", ""),
            "error_types": r.get("error_types", []),
            "severity": r.get("severity", ""),
            "details": r.get("details", ""),
            "score": r.get("score", None),
        } for r in error_results["trained"]
    ],
    "config": {
        "baseline_model": BASELINE_MODEL,
        "lora_path": LORA_WEIGHTS_PATH,
        "vlm_model": VLM_MODEL,
        "seed": SEED,
        "use_4bit": USE_4BIT,
    }
}

with open(os.path.join(OUTPUT_DIR, 'all_qualitative_results.json'), 'w') as f:
    json.dump(results_json, f, indent=2, default=str)

print("\n✓ All results saved!")
print(f"\nFiles in {OUTPUT_DIR}/")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

In [ ]:
# Download results as zip (for Colab)
!cd {OUTPUT_DIR} && zip -r ../qualitative_results.zip .
try:
    from google.colab import files
    files.download('qualitative_results.zip')
except ImportError:
    print("Not running in Colab. Files saved locally in:", OUTPUT_DIR)